<font size = "3">

**(Q1)** Below is the `build_parse_tree` function we saw in class, along with some  example calls to the function.

In [6]:
from assign4_utils import Stack, BinaryTree

def build_parse_tree(fp_expr):
    fp_list = fp_expr.split()
    p_stack = Stack()
    expr_tree = BinaryTree("")
    # p_stack.push(expr_tree)
    p_stack.push(None) # I think this is more clear
    current_tree = expr_tree

    for i in fp_list:
        if i == "(":
            current_tree.insert_left("")
            p_stack.push(current_tree)
            current_tree = current_tree.left_child
        elif i in ["+", "-", "*", "/"]:
            current_tree.key = i
            current_tree.insert_right("")
            p_stack.push(current_tree)
            current_tree = current_tree.right_child
        elif i.isdigit():
              current_tree.key = int(i)
              parent = p_stack.pop()
              current_tree = parent
        elif i == ")":
              current_tree = p_stack.pop()
        else:
              raise ValueError(f"Unknown operator '{i}'")

    return expr_tree

pt = build_parse_tree("( 10 + 3 )")
pt.postorder()

10
3
+


In [13]:
expr = "( ( ( 4 * 3 ) + ( 5 * 2 ) ) / 2 )"
pt = build_parse_tree(expr)
pt.postorder()

4
3
*
5
2
*
+
2
/


<font size = "3">

Modify the function to define `build_parse_tree_v2` which does not require the outermost parentheses in an expression.

In [20]:
def build_parse_tree_v2(fp_expr):
    fp_list = fp_expr.split()
    p_stack = Stack()
    expr_tree = BinaryTree("")
    p_stack.push(None)
    current_tree = expr_tree

    if fp_list[0] != "(" or fp_list[-1] != ")":
        current_tree.insert_left("")
        p_stack.push(current_tree)
        current_tree = current_tree.left_child

    for i in fp_list:
        if i == "(":
            current_tree.insert_left("")
            p_stack.push(current_tree)
            current_tree = current_tree.left_child
        elif i in ["+", "-", "*", "/"]:
            current_tree.key = i
            current_tree.insert_right("")
            p_stack.push(current_tree)
            current_tree = current_tree.right_child
        elif i.isdigit():
              current_tree.key = int(i)
              parent = p_stack.pop()
              current_tree = parent
        elif i == ")":
              current_tree = p_stack.pop()
        else:
              raise ValueError(f"Unknown operator '{i}'")

    return expr_tree

pt = build_parse_tree_v2("( 10 + 3 )")
pt.postorder()


10
3
+


In [19]:
# This should return a tree with 
# the following values in a postorder traversal: 10 3 +
pt = build_parse_tree_v2("10 + 3")
pt.postorder()

10
3
+


In [17]:
# This should return a tree with 
# the following values in a postorder traversal: 4 3 * 5 2 * + 2 /
expr = "( ( 4 * 3 ) + ( 5 * 2 ) ) / 2"
pt = build_parse_tree_v2(expr)
pt.postorder()

4
3
*
5
2
*
+
2
/


<font size = "3">

**(Q2)** We can "invert" the procedure that converts a string into a parse tree using the `print_exp` function:

In [22]:
def print_exp(tree):
    result = ""
    if tree:
        result = "(" + print_exp(tree.left_child)
        result = result + str(tree.key)
        result = result + print_exp(tree.right_child) + ")"
    return result

pt = build_parse_tree("( ( 10 + 5 ) * 3 )")

pt_str = print_exp(pt)

print(pt_str)

(((10)+(5))*(3))


<font size = "3">

Modify the function so that it does not include the extra parentheses around each individual number.

I.e. the modified function would return the string `((10 + 5)*3)`

In [33]:
def print_exp_modified(tree):
    result = ""
    if tree:
        if tree.left_child is None and tree.right_child is None:
            return str(tree.key)
        result = "(" + print_exp_modified(tree.left_child)
        result = result + str(tree.key)
        result = result + print_exp_modified(tree.right_child) + ")"
    return result

pt = build_parse_tree("( ( 10 + 5 ) * 3 )")

pt_str = print_exp_modified(pt)

print(pt_str)

((10+5)*3)


<font size = "3">

**(Q3)** Create a binary heap class with a limited heap size. In other words, the heap only keeps track of the $m$
most important items. If the heap grows in size to more than $m$ items the least important item is dropped (while maintaining the binary heap structure).

Continue to use the *min heap* framework, where the least important item has the largest key value.

In [39]:
class BinaryHeap:
    def __init__(self, m):
        self._heap = []
        self.size = m

    def _perc_up(self, i):
        while (i - 1) // 2 >= 0:
            p = (i - 1) // 2
            if self._heap[i] < self._heap[p]:
                self._heap[i],self._heap[p] = (
                    self._heap[p], self._heap[i]
                )
            else:
                break
            i = p

    def insert(self, item):
        self._heap.append(item)
        self._perc_up(len(self._heap) - 1)
        if len(self._heap) > self.size:
            self.delete_max()
    
    def delete_max(self):
        start = len(self._heap) // 2
        max_idx = start
        for i in range(start, len(self._heap)):
            if self._heap[i] > self._heap[max_idx]:
                max_idx = i
        self._heap[max_idx], self._heap[-1] = self._heap[-1], self._heap[max_idx]
        return self._heap.pop()

    def _get_min_child(self, i):
        if 2 * i + 2 > len(self._heap) - 1: # no right child
            return 2 * i + 1
        if self._heap[2 * i + 1] < self._heap[2 * i + 2]:
            # tie -> take right_child
            return 2 * i + 1
        return 2 * i + 2
    
    def _perc_down(self, i):
        while 2 * i + 1 < len(self._heap):
            sm_child = self._get_min_child(i)
            if self._heap[i] > self._heap[sm_child]:
                self._heap[i], self._heap[sm_child] = (self._heap[sm_child], self._heap[i])
            else:
                break
            i = sm_child

    def delete(self):
        # swap, pop, percolate down
        self._heap[0], self._heap[-1] = self._heap[-1], self._heap[0]
        result = self._heap.pop()
        self._perc_down(0)
        return result

    def heapify(self, not_a_heap):
        # make a copy of the input list
        self._heap = not_a_heap[:]
        # left-most node in second-lowest level
        i = len(self._heap) // 2 - 1
        while i >= 0:
            self._perc_down(i)
            i = i - 1
    
    def is_empty(self):
        return not bool(self._heap)

    def __len__(self):
        return len(self._heap)

    def __str__(self):
        return str(self._heap)

<font size = "3">

**(Q4)** Using the `heapify` function, write a sorting function that can sort a sorted list in $\mathcal{O}(n\log n)$ time.

Keep the following in mind as you design your algorithm:

- The `heapify` method is an $\mathcal{O}(n)$ operation.

- "Percolating" up or down is an $\mathcal{O}(\log n)$ operation.

- Swapping two items in a list is an $\mathcal{O}(1)$ operation.

- Popping from the end of a list is an $\mathcal{O}(1)$ operation.

- Popping from the beginning of a list is an $\mathcal{O}(n)$ operation.

You might find it convenient to write an independent `heapify` function instead of calling the method from a `BinaryHeap` object.

In [ ]:
def heapify_sort(lst):
    h = BinaryHeap(len(lst))
    h.heapify(lst)
    
    list = []
    while not h.is_empty():
        list.append(h.delete())
    
    return list

[1, 2, 3, 4, 7, 8, 9, 10]


<font size = "3">

**(Q5)** Empirically verify that the function you wrote for Question 4 has $\mathcal{O}(n \log n)$ cost by testing it on lists of integers of different sizes.

In [72]:
import time
import random
import math

sizes = [10, 100, 1000, 10000]

for n in sizes:
    lst = [random.randint(0, 500000) for _ in range(n)]

    start = time.time()
    heapify_sort(lst)
    end = time.time()

    print(f"n={n}, time={end - start}, time/(n log n)={(end - start) / (n * math.log(n))}")


n=10, time=4.8160552978515625e-05, time/(n log n)=2.0915862403978553e-06
n=100, time=0.0004649162292480469, time/(n log n)=1.0095527645484697e-06
n=1000, time=0.0058040618896484375, time/(n log n)=8.40224017099759e-07
n=10000, time=0.06318378448486328, time/(n log n)=6.860092236885105e-07
